ADVANCED RUL PREDICTION: Gated Ensemble BiLSTM + Attention (Optimized)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Pre-processing

In [ ]:
import numpy as np
import pandas as pd

from IPython.display import display, HTML
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.io as pio


import seaborn as sns
from importlib import reload
import matplotlib.pyplot as plt
import matplotlib
import warnings

# Configure Jupyter Notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 500)
pd.set_option('display.expand_frame_repr', False)
# pd.set_option('max_colwidth', -1)
display(HTML("<style>div.output_scroll { height: 35em; }</style>"))

reload(plt)
%matplotlib inline
%config InlineBackend.figure_format ='retina'

warnings.filterwarnings('ignore')

# configure plotly graph objects
pio.renderers.default = 'iframe'
# pio.renderers.default = 'vscode'

pio.templates["ck_template"] = go.layout.Template(
    layout_colorway = px.colors.sequential.Viridis,
#     layout_hovermode = 'closest',
#     layout_hoverdistance = -1,
    layout_autosize=False,
    layout_width=800,
    layout_height=600,
    layout_font = dict(family="Calibri Light"),
    layout_title_font = dict(family="Calibri"),
    layout_hoverlabel_font = dict(family="Calibri Light"),
#     plot_bgcolor="white",
)

# pio.templates.default = 'seaborn+ck_template+gridon'
pio.templates.default = 'ck_template+gridon'
# pio.templates.default = 'seaborn+gridon'
# pio.templates

In [ ]:
index_names = ['engine', 'cycle']
setting_names = ['setting_1', 'setting_2', 'setting_3']
sensor_names=[ "(Fan inlet temperature) (◦R)",
"(LPC outlet temperature) (◦R)",
"(HPC outlet temperature) (◦R)",
"(LPT outlet temperature) (◦R)",
"(Fan inlet Pressure) (psia)",
"(bypass-duct pressure) (psia)",
"(HPC outlet pressure) (psia)",
"(Physical fan speed) (rpm)",
"(Physical core speed) (rpm)",
"(Engine pressure ratio(P50/P2)",
"(HPC outlet Static pressure) (psia)",
"(Ratio of fuel flow to Ps30) (pps/psia)",
"(Corrected fan speed) (rpm)",
"(Corrected core speed) (rpm)",
"(Bypass Ratio) ",
"(Burner fuel-air ratio)",
"(Bleed Enthalpy)",
"(Required fan speed)",
"(Required fan conversion speed)",
"(High-pressure turbines Cool air flow)",
"(Low-pressure turbines Cool air flow)" ]
col_names = index_names + setting_names + sensor_names

In [ ]:
df_train = pd.read_csv('drive/MyDrive/CMaps/train_FD004.txt',sep=r'\s+',header=None,index_col=False,names=col_names)
df_test = pd.read_csv('drive/MyDrive/CMaps/test_FD004.txt',sep=r'\s+',header=None,index_col=False,names=col_names)
df_test_RUL = pd.read_csv('drive/MyDrive/CMaps/RUL_FD004.txt',sep=r'\s+',header=None,index_col=False,names=['RUL'])

In [ ]:

# corr = df_train.corr()
# mask = np.triu(np.ones_like(corr, dtype=bool))

# f, ax = plt.subplots(figsize=(20, 20))
# cmap = sns.diverging_palette(230, 10, as_cmap=True)

# # Draw the heatmap with annotation
# sns.heatmap(
#     corr,
#     mask=mask,
#     cmap=cmap,
#     vmax=.3,
#     center=0,
#     square=True,
#     linewidths=.5,
#     cbar_kws={"shrink": .5},
#     annot=True,           # <- Add this line
#     fmt=".1f"             # <- Format the numbers to 2 decimal places
# )

# plt.show()


In [ ]:
# sens_const_values = []
# for feature in list(setting_names + sensor_names):
#     try:
#         if df_train[feature].min()==df_train[feature].max():
#             sens_const_values.append(feature)
#     except:
#         pass

# print(sens_const_values)
# df_train.drop(sens_const_values,axis=1,inplace=True)
# df_test.drop(sens_const_values,axis=1,inplace=True)

In [ ]:
# cor_matrix = df_train.corr().abs()
# upper_tri = cor_matrix.where(np.triu(np.ones(cor_matrix.shape),k=1).astype(np.bool))
# corr_features = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]
# print(corr_features)
# df_train.drop(corr_features,axis=1,inplace=True)
# df_test.drop(corr_features,axis=1,inplace=True)

In [ ]:
features = list(df_train.columns)

In [ ]:
# check for missing data
for feature in features:
    print(feature + " - " + str(len(df_train[df_train[feature].isna()])))

engine - 0
cycle - 0
setting_1 - 0
setting_2 - 0
setting_3 - 0
(Fan inlet temperature) (◦R) - 0
(LPC outlet temperature) (◦R) - 0
(HPC outlet temperature) (◦R) - 0
(LPT outlet temperature) (◦R) - 0
(Fan inlet Pressure) (psia) - 0
(bypass-duct pressure) (psia) - 0
(HPC outlet pressure) (psia) - 0
(Physical fan speed) (rpm) - 0
(Physical core speed) (rpm) - 0
(Engine pressure ratio(P50/P2) - 0
(HPC outlet Static pressure) (psia) - 0
(Ratio of fuel flow to Ps30) (pps/psia) - 0
(Corrected fan speed) (rpm) - 0
(Corrected core speed) (rpm) - 0
(Bypass Ratio)  - 0
(Burner fuel-air ratio) - 0
(Bleed Enthalpy) - 0
(Required fan speed) - 0
(Required fan conversion speed) - 0
(High-pressure turbines Cool air flow) - 0
(Low-pressure turbines Cool air flow) - 0


In [ ]:
# define the maximum life of each engine, as this could be used to obtain the RUL at each point in time of the engine's life
df_train_RUL = df_train.groupby(['engine']).agg({'cycle':'max'})
df_train_RUL.rename(columns={'cycle':'life'},inplace=True)
df_train_RUL.head()

,life
engine,
1,321
2,299
3,307
4,274
5,193


In [ ]:
df_train=df_train.merge(df_train_RUL,how='left',on=['engine'])

In [ ]:
df_train['RUL']=df_train['life']-df_train['cycle']
df_train.drop(['life'],axis=1,inplace=True)

# the RUL prediction is only useful nearer to the end of the engine's life, therefore we put an upper limit on the RUL
# this is a bit sneaky, since it supposes that the test set has RULs of less than this value, the closer you are
# to the true value, the more accurate the model will be
df_train['RUL'][df_train['RUL']>125]=125
df_train.head()

,engine,cycle,setting_1,setting_2,setting_3,(Fan inlet temperature) (◦R),(LPC outlet temperature) (◦R),(HPC outlet temperature) (◦R),(LPT outlet temperature) (◦R),(Fan inlet Pressure) (psia),(bypass-duct pressure) (psia),(HPC outlet pressure) (psia),(Physical fan speed) (rpm),(Physical core speed) (rpm),(Engine pressure ratio(P50/P2),(HPC outlet Static pressure) (psia),(Ratio of fuel flow to Ps30) (pps/psia),(Corrected fan speed) (rpm),(Corrected core speed) (rpm),(Bypass Ratio),(Burner fuel-air ratio),(Bleed Enthalpy),(Required fan speed),(Required fan conversion speed),(High-pressure turbines Cool air flow),(Low-pressure turbines Cool air flow),RUL
0,1,1,42.0049,0.8400,100.0,445.00,549.68,1343.43,1112.93,3.91,5.70,137.36,2211.86,8311.32,1.01,41.69,129.78,2387.99,8074.83,9.3335,0.02,330,2212,100.00,10.62,6.3670,125
1,1,2,20.0020,0.7002,100.0,491.19,606.07,1477.61,1237.50,9.35,13.61,332.10,2323.66,8713.60,1.07,43.94,312.59,2387.73,8046.13,9.1913,0.02,361,2324,100.00,24.37,14.6552,125
2,1,3,42.0038,0.8409,100.0,445.00,548.95,1343.12,1117.05,3.91,5.69,138.18,2211.92,8306.69,1.01,41.66,129.62,2387.97,8066.62,9.4007,0.02,329,2212,100.00,10.48,6.4213,125
3,1,4,42.0000,0.8400,100.0,445.00,548.70,1341.24,1118.03,3.91,5.70,137.98,2211.88,8312.35,1.02,41.68,129.80,2388.02,8076.05,9.3369,0.02,328,2212,100.00,10.54,6.4176,125
4,1,5,25.0063,0.6207,60.0,462.54,536.10,1255.23,1033.59,7.05,9.00,174.82,1915.22,7994.94,0.93,36.48,164.11,2028.08,7865.80,10.8366,0.02,305,1915,84.93,14.03,8.6754,125


In [ ]:
def create_sequences(df, window_size, stride, engine_col='engine'):
    sequences, targets, engine_ids = [], [], []

    for engine_id in df[engine_col].unique():
        engine_data = df[df[engine_col] == engine_id].sort_values('cycle')
        feature_cols = [col for col in engine_data.columns if col not in ['engine', 'cycle', 'RUL']]

        values = engine_data[feature_cols].values
        rul_values = engine_data['RUL'].values

        for i in range(0, len(values) - window_size + 1, stride):
            sequences.append(values[i:i + window_size])
            targets.append(rul_values[i + window_size - 1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)

# Scale features
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
feature_cols = [col for col in df_train.columns if col not in ['engine', 'cycle', 'RUL']]
df_train[feature_cols] = scaler.fit_transform(df_train[feature_cols])
df_test[feature_cols] = scaler.transform(df_test[feature_cols])

In [ ]:
pip install tensorflow

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, regularizers
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

In [ ]:
np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# ============================================================
# ADVANCED RUL PREDICTION: Gated Ensemble BiLSTM + Attention (Optimized)
# ============================================================

import tensorflow as tf
from tensorflow.keras import layers, models, regularizers, callbacks, optimizers
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# MODIFIED create_sequences function to handle missing 'RUL' column for test set
def create_sequences(df, window_size, stride, engine_col='engine'):
    sequences, targets, engine_ids = [], [], []

    # Check if 'RUL' column exists in the DataFrame
    has_rul = 'RUL' in df.columns

    for engine_id in df[engine_col].unique():
        engine_data = df[df[engine_col] == engine_id].sort_values('cycle')
        feature_cols = [col for col in engine_data.columns if col not in ['engine', 'cycle', 'RUL']]

        values = engine_data[feature_cols].values

        if has_rul:
            rul_values = engine_data['RUL'].values
        else:
            # If RUL column is not present (e.g., for test data), create a dummy array.
            # These 'targets' will be discarded for X_test anyway.
            rul_values = np.array([])

        # Ensure there are enough data points for the window
        if len(values) < window_size:
            continue # Skip engines that are too short for a single sequence

        for i in range(0, len(values) - window_size + 1, stride):
            sequences.append(values[i:i + window_size])
            if has_rul: # Only append RUL target if RUL column exists
                targets.append(rul_values[i + window_size - 1])
            engine_ids.append(engine_id)

    return np.array(sequences), np.array(targets), np.array(engine_ids)


CONFIG = {
    'WINDOW_SIZE': 60,
    'STRIDE': 2,
    'LSTM_UNITS': [128, 64],
    'DENSE_UNITS': 80,
    'DROPOUT_RATE': 0.3,
    'L2_REG': 0.001,
    'BATCH_SIZE': 32,
    'EPOCHS': 200,
    'LEARNING_RATE': 0.0005,
    'PATIENCE': 35,
    'N_SPLITS': 5
}

print("\nOptimized Configuration:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")


X_train, y_train, train_engine_ids = create_sequences(
    df_train, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE']
)
X_test, _, test_engine_ids = create_sequences(
    df_test, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE']
)

print(f"Training sequences: {X_train.shape}")
print(f"Test sequences: {X_test.shape}")


# ============================================================
# Custom Layers
# ============================================================

class GatingLayer(layers.Layer):
    def __init__(self, num_branches, **kwargs):
        super(GatingLayer, self).__init__(**kwargs)
        self.num_branches = num_branches

    def build(self, input_shape):
        self.gate_weights = self.add_weight(
            shape=(input_shape[0][-1], self.num_branches),
            initializer='glorot_uniform',
            trainable=True
        )
        self.gate_bias = self.add_weight(
            shape=(self.num_branches,),
            initializer='zeros',
            trainable=True
        )

    def call(self, inputs):
        input_stats = tf.reduce_mean(inputs[0], axis=1)
        gate_scores = tf.nn.softmax(tf.matmul(input_stats, self.gate_weights) + self.gate_bias, axis=-1)
        gate_scores = tf.expand_dims(tf.expand_dims(gate_scores, 1), -1)
        stacked = tf.stack(inputs, axis=2)
        gated = stacked * gate_scores
        fused = tf.reduce_sum(gated, axis=2)
        return fused


class AttentionLayer(layers.Layer):
    """Attention mechanism to highlight critical timesteps after fusion."""
    def __init__(self, **kwargs):
        super(AttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(shape=(input_shape[-1], input_shape[-1]),
                                 initializer="glorot_uniform", trainable=True)
        self.b = self.add_weight(shape=(input_shape[-1],),
                                 initializer="zeros", trainable=True)
        self.u = self.add_weight(shape=(input_shape[-1],),
                                 initializer="glorot_uniform", trainable=True)

    def call(self, x):
        u_t = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        attn_scores = tf.nn.softmax(tf.tensordot(u_t, self.u, axes=1), axis=1)
        attn_scores = tf.expand_dims(attn_scores, -1)
        context = tf.reduce_sum(x * attn_scores, axis=1)
        return context


# ============================================================
# Optimized Gated BiLSTM + Attention Model
# ============================================================

def build_gated_attention_bilstm(input_shape, config):
    inputs = layers.Input(shape=input_shape, name='input')

    # Parallel branches: LSTM, GRU, CNN
    branch_lstm = layers.LSTM(64, return_sequences=True,
                              kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_lstm = layers.LayerNormalization()(branch_lstm)

    branch_gru = layers.GRU(64, return_sequences=True,
                            kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_gru = layers.LayerNormalization()(branch_gru)

    branch_cnn = layers.Conv1D(64, 5, padding='same', activation='relu',
                               kernel_regularizer=regularizers.l2(config['L2_REG']))(inputs)
    branch_cnn = layers.Conv1D(64, 3, padding='same', activation='relu',
                               kernel_regularizer=regularizers.l2(config['L2_REG']))(branch_cnn)
    branch_cnn = layers.LayerNormalization()(branch_cnn)

    fused = GatingLayer(num_branches=3)([branch_lstm, branch_gru, branch_cnn])
    fused = layers.Dropout(config['DROPOUT_RATE'])(fused)

    # BiLSTM layers for sequential modeling
    x = layers.Bidirectional(
        layers.LSTM(config['LSTM_UNITS'][0],
                    return_sequences=True,
                    kernel_regularizer=regularizers.l2(config['L2_REG'])),
        name='bilstm_1'
    )(fused)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    x = layers.Bidirectional(
        layers.LSTM(config['LSTM_UNITS'][1],
                    return_sequences=True,
                    kernel_regularizer=regularizers.l2(config['L2_REG'])),
        name='bilstm_2'
    )(x)
    x = layers.LayerNormalization()(x)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    # Attention layer
    context = AttentionLayer()(x)

    # Dense head
    x = layers.Dense(config['DENSE_UNITS'], activation='relu')(context)
    x = layers.Dropout(config['DROPOUT_RATE'])(x)

    outputs = layers.Dense(1, activation='linear', name='rul_output')(x)

    model = models.Model(inputs, outputs, name='Gated_Attention_BiLSTM')

    opt = optimizers.Adam(learning_rate=config['LEARNING_RATE'])
    model.compile(
        optimizer=opt,
        loss=tf.keras.losses.Huber(delta=1.0),
        metrics=['mae', 'mse']
    )
    return model


# ============================================================
# Train with Advanced Callbacks
# ============================================================

def train_model_with_cv(X_train, y_train, engine_ids, input_shape, config, n_splits=5):
    from sklearn.model_selection import GroupKFold

    gkf = GroupKFold(n_splits=n_splits)
    results, models_list = [], []

    for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_train, y_train, engine_ids), 1):
        print(f"\n{'='*25} FOLD {fold}/{n_splits} {'='*25}")
        X_tr, X_val = X_train[tr_idx], X_train[val_idx]
        y_tr, y_val = y_train[tr_idx], y_train[val_idx]

        model = build_gated_attention_bilstm(input_shape, config)

        cb = [
            callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6),
            callbacks.EarlyStopping(monitor='val_loss', patience=config['PATIENCE'],
                                    restore_best_weights=True),
            callbacks.ModelCheckpoint(f"best_fold_{fold}.h5", save_best_only=True)
        ]

        history = model.fit(
            X_tr, y_tr,
            validation_data=(X_val, y_val),
            epochs=config['EPOCHS'],
            batch_size=config['BATCH_SIZE'],
            verbose=1,
            callbacks=cb
        )

        val_loss, val_mae, val_mse = model.evaluate(X_val, y_val, verbose=0)
        val_rmse = np.sqrt(val_mse)

        y_pred = model.predict(X_val, verbose=0).flatten()
        val_r2 = r2_score(y_val, y_pred)

        results.append({'fold': fold, 'mae': val_mae, 'rmse': val_rmse, 'r2': val_r2})
        models_list.append(model)

        print(f"Fold {fold} Results -> MAE: {val_mae:.4f}, RMSE: {val_rmse:.4f}, R²: {val_r2:.4f}")

    print("\n=== Cross-Validation Summary ===")
    avg_mae = np.mean([r['mae'] for r in results])
    avg_rmse = np.mean([r['rmse'] for r in results])
    avg_r2 = np.mean([r['r2'] for r in results])
    print(f"MAE: {avg_mae:.4f}, RMSE: {avg_rmse:.4f}, R²: {avg_r2:.4f}")

    best_model_idx = np.argmax([r['r2'] for r in results])
    print(f"\nBest Fold: {best_model_idx + 1}")
    return models_list[best_model_idx], results


# ============================================================
# Execute Training
# ============================================================

input_shape = (X_train.shape[1], X_train.shape[2])
best_model, cv_results = train_model_with_cv(X_train, y_train, train_engine_ids, input_shape, CONFIG, CONFIG['N_SPLITS'])

# ============================================================
# Evaluate on Test Set
# ============================================================

y_test_pred_all = best_model.predict(X_test, verbose=0).flatten()
y_test_pred_all = np.clip(y_test_pred_all, 0, None)

# Take last window per engine
unique_engines = sorted(set(test_engine_ids))
y_true, y_pred = [], []
for eng_id in unique_engines:
    mask = test_engine_ids == eng_id
    y_pred.append(y_test_pred_all[mask][-1])
    y_true.append(df_test_RUL.iloc[int(eng_id) - 1]['RUL'])

y_true, y_pred = np.array(y_true), np.array(y_pred)
test_mae = mean_absolute_error(y_true, y_pred)
test_rmse = np.sqrt(mean_squared_error(y_true, y_pred))
test_r2 = r2_score(y_true, y_pred)

print(f"\n✅ Optimized Test Results:")
print(f"  MAE:  {test_mae:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

if test_r2 >= 0.85:
    print("\n🏆 EXCELLENT PERFORMANCE ACHIEVED!")



Optimized Configuration:
  WINDOW_SIZE: 40
  STRIDE: 2
  LSTM_UNITS: [128, 64]
  DENSE_UNITS: 80
  DROPOUT_RATE: 0.3
  L2_REG: 0.001
  BATCH_SIZE: 32
  EPOCHS: 200
  LEARNING_RATE: 0.0005
  PATIENCE: 35
  N_SPLITS: 5
Training sequences: (25826, 40, 24)
Test sequences: (15929, 40, 24)

========================= FOLD 1/5 =========================
Epoch 1/200
646/646 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 50.8152 - mae: 50.4979 - mse: 3877.2234

646/646 ━━━━━━━━━━━━━━━━━━━━ 37s 36ms/step - loss: 50.7904 - mae: 50.4732 - mse: 3874.3933 - val_loss: 17.4876 - val_mae: 17.3010 - val_mse: 531.1335 - learning_rate: 5.0000e-04
Epoch 2/200
645/646 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - loss: 20.3304 - mae: 20.1569 - mse: 722.7783

646/646 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 20.3298 - mae: 20.1564 - mse: 722.7396 - val_loss: 16.2561 - val_mae: 16.1149 - val_mse: 480.7980 - learning_rate: 5.0000e-04
Epoch 3/200
646/646 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 19.3134 - mae: 19.1802 - mse: 661.5758

646/646 ━━━━━━━━━━━━━━━━━━━━ 43s 34ms/step - loss: 19.3130 - mae: 19.1798 - mse: 661.5458 - val_loss: 15.8711 - val_mae: 15.7690 - val_mse: 435.7366 - learning_rate: 5.0000e-04
Epoch 4/200
646/646 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 18.0746 - mae: 17.9787 - mse: 581.7769 - val_loss: 16.1993 - val_mae: 16.1287 - val_mse: 471.3832 - learning_rate: 5.0000e-04
Epoch 5/200
646/646 ━━━━━━━━━━━━━━━━━━━━ 24s 37ms/step - loss: 17.2386 - mae: 17.1717 - mse: 536.9205 - val_loss: 16.2754 - val_mae: 16.2262 - val_mse: 464.8203 - learning_rate: 5.0000e-04
Epoch 6/200
645/646 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 16.3903 - mae: 16.3419 - mse: 491.1798

646/646 ━━━━━━━━━━━━━━━━━━━━ 39s 34ms/step - loss: 16.3898 - mae: 16.3414 - mse: 491.1417 - val_loss: 14.3320 - val_mae: 14.2978 - val_mse: 365.0586 - learning_rate: 5.0000e-04
Epoch 7/200
646/646 ━━━━━━━━━━━━━━━━━━━━ 20s 31ms/step - loss: 15.8148 - mae: 15.7810 - mse: 457.8661 - val_loss: 14.8094 - val_mae: 14.7845 - val_mse: 421.8791 - learning_rate: 5.0000e-04
Epoch 8/200
645/646 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - loss: 15.5456 - mae: 15.5170 - mse: 442.4420

...

647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - loss: 17.8583 - mae: 17.7452 - mse: 576.4191 - val_loss: 16.5147 - val_mae: 16.4402 - val_mse: 515.0837 - learning_rate: 5.0000e-04
Epoch 4/200
646/647 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - loss: 17.0560 - mae: 16.9861 - mse: 532.4623

647/647 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - loss: 17.0552 - mae: 16.9853 - mse: 532.4174 - val_loss: 14.7019 - val_mae: 14.6576 - val_mse: 458.2019 - learning_rate: 5.0000e-04
Epoch 5/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 16.2011 - mae: 16.1603 - mse: 484.7568

647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - loss: 16.2009 - mae: 16.1600 - mse: 484.7416 - val_loss: 13.5415 - val_mae: 13.5164 - val_mse: 390.9541 - learning_rate: 5.0000e-04
Epoch 6/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 15.4120 - mae: 15.3900 - mse: 436.6396 - val_loss: 13.6321 - val_mae: 13.6205 - val_mse: 384.5920 - learning_rate: 5.0000e-04
Epoch 7/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 14.8634 - mae: 14.8521 - mse: 414.1678

647/647 ━━━━━━━━━━━━━━━━━━━━ 23s 36ms/step - loss: 14.8634 - mae: 14.8521 - mse: 414.1667 - val_loss: 12.3444 - val_mae: 12.3421 - val_mse: 358.9181 - learning_rate: 5.0000e-04
Epoch 8/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - loss: 14.3669 - mae: 14.3634 - mse: 383.1433 - val_loss: 13.2289 - val_mae: 13.2314 - val_mse: 368.2870 - learning_rate: 5.0000e-04
Epoch 9/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 14.0172 - mae: 14.0179 - mse: 371.5515 - val_loss: 14.2787 - val_mae: 14.2860 - val_mse: 414.4207 - learning_rate: 5.0000e-04
Epoch 10/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - loss: 13.3745 - mae: 13.3779 - mse: 335.6632 - val_loss: 12.6730 - val_mae: 12.6785 - val_mse: 357.5479 - learning_rate: 5.0000e-04
Epoch 11/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 13.0806 - mae: 13.0823 - mse: 318.1779 - val_loss: 13.2681 - val_mae: 13.2687 - val_mse: 382.4723 - learning_rate: 5.0000e-04
Epoch 12/200
646/647 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - loss: 12

647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - loss: 12.4936 - mae: 12.4890 - mse: 295.0843 - val_loss: 12.0203 - val_mae: 12.0152 - val_mse: 381.6892 - learning_rate: 5.0000e-04
Epoch 13/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 21s 32ms/step - loss: 12.5038 - mae: 12.4938 - mse: 293.8611 - val_loss: 12.4433 - val_mae: 12.4217 - val_mse: 383.4427 - learning_rate: 5.0000e-04
Epoch 14/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - loss: 12.0348 - mae: 12.0164 - mse: 275.2196 - val_loss: 13.5684 - val_mae: 13.5478 - val_mse: 406.4919 - learning_rate: 5.0000e-04
Epoch 15/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 22s 34ms/step - loss: 11.6044 - mae: 11.5785 - mse: 252.3595 - val_loss: 12.0695 - val_mae: 12.0368 - val_mse: 362.6602 - learning_rate: 5.0000e-04
Epoch 16/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 42s 35ms/step - loss: 11.2650 - mae: 11.2285 - mse: 237.9103 - val_loss: 14.1749 - val_mae: 14.1377 - val_mse: 384.5380 - learning_rate: 5.0000e-04
Epoch 17/200
647/647 ━━━━━━━━━━━━━━━━━━━━ 21s 33ms/step - loss: